# Exercise 1: Creating a Movie Database Extracted from an API

In [25]:
# Libraries for HTTP requests and data manipulation
import requests
import pandas as pd 
from IPython.display import HTML

# MySQL database connection
import mysql.connector
from mysql.connector import Error

# Used to clean API values
import numpy as np

# Environment variables (passwords, sensitive config)
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

# Suppress warning messages to keep the output clean
import warnings
warnings.filterwarnings("ignore")

## Phase 1: Movie Data Extraction

In [26]:
# Fetches movie data from an API and returns a pandas DataFrame
def api_requests(url_api_movies):

    try:
        # Send a GET request to the API URL 
        movie_data = requests.get(url_api_movies)
        if movie_data.status_code == 200:
            print ("API connected")
            # Convert the JSON response into a pandas DataFrame
            df_movies = pd.DataFrame(movie_data.json())
            return df_movies
        else:
            # Handle cases where the server responds with an error code
            print ("API failed")

    # Handle connection-related errors (e.g., DNS failure, refused connection)        
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # Handle requests that exceed the timeout limit
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # Handle any other ambiguous exception that occurs while handling a request
    except requests.exceptions.RequestException as e:
        print (e)        
   

In [27]:
# Fetch data from the URL and store the returned DataFrame in df_movies
df_movies = api_requests("https://beta.adalab.es/resources/apis/pelis/pelis.json")
HTML(df_movies.to_html(index=False))

API connected


id,titulo,año,duracion,genero,adultos,subtitulos
1,The Godfather,1972,175,Crimen,False,"[es, en]"
2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
5,The Dark Knight,2008,152,Acción,False,"[es, en]"
6,Fight Club,1999,139,Drama,True,"[es, en]"
7,Inception,2010,148,Ciencia ficción,False,"[es, en, de]"
8,The Matrix,1999,136,Ciencia ficción,False,"[es, en]"
9,The Shawshank Redemption,1994,142,Drama,False,"[es, en]"
10,Interstellar,2014,169,Ciencia ficción,False,"[es, en]"


## Phase 2: Database Creation

In [28]:
# Establishes a connection to the local MySQL server
def connection_mysql():

    try:
        # Using connector from the library
        connection = mysql.connector.connect(
            host = "127.0.0.1", 
            user= "root",
            password = password_sql,  # Uses the local environment password variable, not added in GitHUB
            # Note: We are creating a new database, so is not included
        )
        print("Connection successful")
        return connection
    
    # Handle any database-related errors that occur during connection
    except Error as e:
        print (f"An error has occurred: {e}")

In [29]:
# Call the function to initialize the database connection and store it
connection = connection_mysql()

Connection successful


In [30]:
# Define database and table names as variables for easier configuration and reuse
db_name = "movies_final_exercise" 
table_movies = "movies_info"
table_subtitles = "subtitles_info"
table_movie_subtitles = "movie_subtitles"

In [31]:
# Creates a new database if it does not already exist
def create_database(db_name):
    
    try:
        # Safely open the cursor using a context manager to ensure automatic closure
        with connection.cursor() as cursor:
            query = f"CREATE DATABASE IF NOT EXISTS {db_name}"
            # Execute the database creation query
            cursor.execute(query)
            print ("Query successful")

    # Catch any SQL execution errors
    except Error as e:
        print (f"Error creating database: {e}")
        

In [32]:
# Using the function we create the database using the variable 
create_database(db_name)

Query successful


## Phase 3: Inserting Data into the Database

In [33]:
# Note: we study the DF columns and types. It is important to notice that "subtitulos" contains a list.
df_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          100 non-null    int64 
 1   titulo      100 non-null    str   
 2   año         100 non-null    int64 
 3   duracion    100 non-null    int64 
 4   genero      100 non-null    str   
 5   adultos     100 non-null    bool  
 6   subtitulos  100 non-null    object
dtypes: bool(1), int64(3), object(1), str(2)
memory usage: 4.9+ KB


In [34]:
# Creates a table inside the designated database if it does not exist
def create_table_generic(db_name, table_name, table_schema):
    
    try:
        # Safely open the cursor using a context manager to ensure automatic closure
        with connection.cursor() as cursor:
            cursor.execute(f"USE {db_name};")
            # Define the SQL query for generic table
            query = f''' CREATE TABLE IF NOT EXISTS {table_name} ({table_schema});'''
            # Execute the table creation query
            cursor.execute(query)
            print ("Query creation successful")
    
    # Catch any SQL execution errors
    except Error as e:
        print (f"Error creating table: {e}")


In [35]:
# Schemas definition for diferent tables, so it is reusable and easy to change if needed:
schema_movies = '''movie_id INT PRIMARY KEY AUTO_INCREMENT,
    title VARCHAR(100) NOT NULL,
    year YEAR,
    runtime INT,
    genre VARCHAR(30),
    is_adult BOOL
'''

schema_subtitles = '''
    subtitle_id INT PRIMARY KEY AUTO_INCREMENT,
    language_code VARCHAR(10) UNIQUE NOT NULL
'''

schema_movie_subtitles = '''
    movie_id INT,
    subtitle_id INT,
    PRIMARY KEY (movie_id, subtitle_id),
    FOREIGN KEY (movie_id) REFERENCES movies_info(movie_id) ON DELETE CASCADE,
    FOREIGN KEY (subtitle_id) REFERENCES subtitles_info(subtitle_id) ON DELETE CASCADE
'''

# Call the generic function for each table, includinf the variables previously configurated
create_table_generic(db_name, table_movies, schema_movies)
create_table_generic(db_name, table_subtitles, schema_subtitles)
create_table_generic(db_name, table_movie_subtitles, schema_movie_subtitles)

Query creation successful
Query creation successful
Query creation successful


In [36]:
# Stablishing a function that can delete tables if is needed. 
def erase_table(db_name, table_name):

    try:

        # Check if the user confirmed the operation
        user_respond = input (f"Table '{table_name}' will be deleted. Are you sure? (Y/N):").upper()
        if user_respond == "Y":
           # Safely open the cursor using a context manager to ensure automatic closure
            with connection.cursor() as cursor:
                # Select the database passed as an argument
                cursor.execute(f"USE {db_name};")
                # Define the SQL query to drop the table if it exists
                query = f'''DROP TABLE IF EXISTS {table_name}'''
                # Execute the drop table query
                cursor.execute(query)
                connection.commit()
                print ("Table deleted successfully")
        else: 
            print ("Operation cancelled")

    # Catch any SQL execution errors
    except Error as e:
        print (f"Error deleting table: {e}")

In [37]:
erase_table(db_name, table_movie_subtitles)
#Consider foraign keys when deteling

Table deleted successfully


In [38]:
create_table_generic(db_name, table_movie_subtitles, schema_movie_subtitles)

Query creation successful


In [39]:
# Inserts multiple rows of clean data from a DataFrame specifically into -table_movies-
def insert_movie_data(db_name):

    try:
        # Safely open the cursor using a context manager to ensure automatic closure
        with connection.cursor() as cursor:
            # Select the database passed as an argument
            cursor.execute(f"USE {db_name};")
            # Define the parameterized SQL query for batch insertion
            query = f'''INSERT INTO {table_movies} (title, year, runtime, 
                    genre, is_adult)
                    VALUES (%s, %s, %s, %s, %s) '''
            # Clean the DataFrame by replacing all variations of NaN with None (SQL NULL). 
            # String data and reorder columns
            df_clean = df_movies.replace({np.nan: None, 'nan': None, 'Nan': None})
            df_clean = df_clean[["titulo", "año", "duracion", "genero", "adultos"]]

            # Convert the DataFrame rows into a list of tuples for the database
            values = df_clean.values.tolist() 

            # Execute the batch insertion of all rows at once
            cursor.executemany(query, values)
            connection.commit()  
            print(f"Successfully inserted {cursor.rowcount} new records")
    
    # Catch any SQL execution errors
    except Error as e:
        print (f"Error inserting data: {e}")
        

In [40]:
insert_movie_data(db_name)

Successfully inserted 100 new records


In [41]:
# Inserts multiple rows of clean data from a DataFrame specifically into -table_subtitles-
def insert_subtitles_data(db_name):

    try:
        # Safely open the cursor using a context manager to ensure automatic closure
        with connection.cursor() as cursor:
            # Select the database passed as an argument
            cursor.execute(f"USE {db_name};")
            # Define the parameterized SQL query for batch insertion
            query = f'''INSERT INTO {table_subtitles} (language_code)
                    VALUES (%s) '''

            # Convert the DataFrame rows into a list of tuples for the database
            df_subtitles = df_movies['subtitulos'].explode()
            values = df_subtitles.dropna().unique().tolist()

            values_tuples = []
            for language in values:
                values_tuples.append((language,))

            # Execute the batch insertion of all rows at once
            cursor.executemany(query, values_tuples)
            connection.commit()  
            print(f"Successfully inserted {cursor.rowcount} new records")
        
    # Catch any SQL execution errors
    except Error as e:
        print (f"Error inserting data: {e}")

In [42]:
insert_subtitles_data(db_name)

Successfully inserted 8 new records


In [43]:
# Inserts multiple rows of clean data from a DataFrame specifically into -table_movie_subtitles-
def insert_movie_subtitles_data(db_name):

    try:
        # Safely open the cursor using a context manager to ensure automatic closure
        with connection.cursor() as cursor:
            # Select the database passed as an argument
            cursor.execute(f"USE {db_name};")

            # Fetch the registered subtitles from the database to create a translator mapping
            cursor.execute(f"SELECT subtitle_id, language_code FROM {table_subtitles};")
            result = cursor.fetchall()

            # Create a dictionary to map language codes to their respective database IDs
            subtitle_dict = {}
            for id, language in result:
                subtitle_dict[language] = id

            # Generate the pairs of IDs matching each movie with its corresponding subtitles
            values_tuples = []
            for index, row in df_movies.iterrows():
                movie_id = row['id']
                sub_list = row['subtitulos']
                
                # Check if the row contains a valid list of subtitles
                for language in sub_list:
                    if language in subtitle_dict:
                        subtitle_id = subtitle_dict[language]
                        values_tuples.append((movie_id, subtitle_id))
            
            # Define the parameterized SQL query for batch insertion into the bridge table
            query = f"INSERT INTO {table_movie_subtitles} (movie_id, subtitle_id) VALUES (%s, %s);"


            # Execute the batch insertion of all rows at once
            cursor.executemany(query, values_tuples)
            connection.commit()  
            print(f"Successfully inserted {cursor.rowcount} new records")
        
    # Catch any SQL execution errors
    except Error as e:
        print (f"Error inserting data: {e}")

In [44]:
insert_movie_subtitles_data(db_name)

Successfully inserted 211 new records


In [ ]:
## We commented out the function used in the first version to prevent its use. ##

# Inserts multiple rows of clean data from a DataFrame into the specified table
#def insert_data(db_name, table_name):

#    try:
        # Safely open the cursor using a context manager to ensure automatic closure
#        with connection.cursor() as cursor:
            # Select the database passed as an argument
#            cursor.execute(f"USE {db_name};")
            # Define the parameterized SQL query for batch insertion
#            query = f'''INSERT INTO {table_name} (title, year, runtime, 
#                    genre, is_adult, subtitles)
  #                  VALUES (%s, %s, %s, %s, %s, %s) '''
            # Clean the DataFrame by replacing all variations of NaN with None (SQL NULL). 
            # String data and reorder columns
  #          df_clean = df_movies.replace({np.nan: None, 'nan': None, 'Nan': None})
  #          df_clean["subtitulos"] = df_clean["subtitulos"].astype(str)
  #          df_clean = df_clean[["titulo", "año", "duracion", "genero", "adultos", "subtitulos"]]

            # Convert the DataFrame rows into a list of tuples for the database
  #          values = df_clean.values.tolist() 

            # Execute the batch insertion of all rows at once
  #          cursor.executemany(query, values)
  #          connection.commit()  
  #          print(f"Successfully inserted {cursor.rowcount} new records")
    
    # Catch any SQL execution errors
#    except Error as e:
#        print (f"Error inserting data: {e}")
        

In [ ]:
# insert_data(db_name, table_movies)

## Phase 4: Querying the Data

In [45]:
# Executes a general SQL query and returns the results as a pandas DataFrame
def general_query(db_name, sql_query):
    
    try:
        # Safely open the cursor using a context manager to ensure automatic closure
        with connection.cursor() as cursor:
            # Select the database passed as an argument
            cursor.execute(f"USE {db_name};")
            # Execute the SQL query and fetch the results directly into a DataFrame
            df_query = pd.read_sql(sql_query, connection)
            return df_query
    
    # Catch any SQL execution errors
    except Error as e:
        print (f"Error inserting data: {e}")
    

### 1. How many movies have a runtime longer than 120 minutes?

In [46]:
# Define the SQL query required and execute it using the general function
query_runtime = f'''SELECT * 
            FROM {table_movies}
            WHERE runtime > 120
            ORDER BY runtime DESC; '''

df_query = general_query(db_name, query_runtime)
HTML(df_query.to_html(index=False))

movie_id,title,year,runtime,genre,is_adult
2,The Godfather Part II,1974,202,Crimen,0
23,The Lord of the Rings: The Return of the King,2003,201,Fantasía,0
77,Schindler's List,1993,195,Drama,1
12,Titanic,1997,195,Romance,0
38,The Green Mile,1999,189,Drama,1
45,Avengers: Endgame,2019,181,Acción,0
52,The Wolf of Wall Street,2013,180,Biografía,1
22,The Lord of the Rings: The Two Towers,2002,179,Fantasía,0
21,The Lord of the Rings: The Fellowship of the Ring,2001,178,Fantasía,0
1,The Godfather,1972,175,Crimen,0


### 2. How many movies include Spanish subtitles?

In [49]:
# Define the SQL query required and execute it using the general function
query_subtitles = f'''SELECT language_code AS language, 
            COUNT(movie_id) AS total_spanish 
            FROM {table_subtitles} s 
            INNER JOIN {table_movie_subtitles} ms ON ms.subtitle_id = s.subtitle_id 
            WHERE language_code = 'es';'''



df_query = general_query(db_name, query_subtitles)
HTML(df_query.to_html(index=False))

language,total_spanish
es,100


### 3. How many movies have adult content?

In [50]:
# Define the SQL query required and execute it using the general function
query_adult = f'''SELECT COUNT(is_adult) AS rating_adult
            FROM {table_movies}
            WHERE is_adult = TRUE;'''

df_query = general_query(db_name, query_adult)
HTML(df_query.to_html(index=False))


rating_adult
47


### 4. What is the oldest movie registered in the database?

In [51]:
# Define the SQL query required and execute it using the general function
query_oldest = f'''SELECT *
            FROM {table_movies}
            WHERE year = (SELECT MIN(year)
                        FROM {table_movies});'''

df_query = general_query(db_name, query_oldest)
HTML(df_query.to_html(index=False))

movie_id,title,year,runtime,genre,is_adult
72,Citizen Kane,1941,119,Drama,0


### 5. Show the average movie runtime grouped by genre.

In [52]:
# Define the SQL query required and execute it using the general function
query_runtime_genre = f'''SELECT genre,
                round(AVG(runtime),2) AS runtime_avg
            FROM {table_movies}
            GROUP BY genre
            ORDER BY runtime_avg;'''

df_query = general_query(db_name, query_runtime_genre)
HTML(df_query.to_html(index=False))

genre,runtime_avg
Animación,103.00
Terror,116.57
Comedia,117.00
Suspense,120.00
Thriller,121.67
Drama,126.26
Musical,128.00
Aventura,133.00
Ciencia ficción,136.31
Acción,139.44


### 6. How many movies have been registered per year? Sort from highest to lowest.

In [53]:
# Define the SQL query required and execute it using the general function
query_per_year = f'''SELECT year,
                COUNT(*) AS registered_per_year
            FROM {table_movies}
            GROUP BY year
            ORDER BY registered_per_year DESC;'''

df_query = general_query(db_name, query_per_year)
HTML(df_query.to_html(index=False))

year,registered_per_year
2001,5
2013,4
1994,4
2008,4
1999,4
2017,4
2010,3
1998,3
2014,3
2000,3


### 7. Which year has the highest number of movies in the database?

In [54]:
# Define the SQL query required and execute it using the general function
query_highest_year = f'''SELECT year,
                COUNT(*) AS movies_per_year
            FROM {table_movies}
            GROUP BY year
            ORDER BY movies_per_year DESC
            LIMIT 1;'''

df_query = general_query(db_name, query_highest_year)
HTML(df_query.to_html(index=False))

year,movies_per_year
2001,5


### 8. Get a list of all genres and the number of movies corresponding to each one.

In [55]:
# Define the SQL query required and execute it using the general function
query_per_genre = f'''SELECT genre,
                COUNT(*) AS movies_per_genre
            FROM {table_movies}
            GROUP BY genre
            ORDER BY genre ASC;'''

df_query = general_query(db_name, query_per_genre)
HTML(df_query.to_html(index=False))

genre,movies_per_genre
Acción,9
Animación,9
Aventura,3
Bélico,2
Biografía,3
Ciencia ficción,13
Comedia,1
Crimen,7
Drama,27
Fantasía,5


### 9. Show all movies whose title contains the word "Godfather".

In [56]:
# Define the word we are going to search
searching_word = "Godfather"

# Define the SQL query required and execute it using the general function
query_word = f'''SELECT * 
            FROM {table_movies}
            WHERE title LIKE "%{searching_word}%";'''

df_query = general_query(db_name, query_word)
HTML(df_query.to_html(index=False))

movie_id,title,year,runtime,genre,is_adult
1,The Godfather,1972,175,Crimen,0
2,The Godfather Part II,1974,202,Crimen,0


## CLOSING CONNECTION TO MYSQL

In [57]:
# Safely closes the connection to the MySQL server
def close_mysql():

    try:
        # Close the active database connection
        connection.close()
        print("Disconnected successfully")
    
    # Handle any database-related errors that occur during disconnection
    except Error as e:
        print (f"An error has occurred: {e}")


In [58]:
close_mysql()

Disconnected successfully
